In [1]:
pip install pandas sqlalchemy pyodbc matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from sqlalchemy import text
from database import engine
import numpy as np

print("Konekcija uspešna!")

--- USPEŠNO KONFIGURISAN ENGINE SA DRAJVEROM: {SQL Server} ---
Konekcija uspešna!


In [3]:
sql_query = text("""
    DECLARE @MaxDate DATETIME = (SELECT MAX(Ts) FROM dbo.MeterReadTfes);
    
    SELECT 
        m.Mid AS meter_id, 
        m.Val AS value, 
        m.Ts  AS timestamp
    FROM dbo.MeterReadTfes m
    JOIN dbo.Meters me ON m.Mid = me.Id
    JOIN dbo.DistributionSubstation ds ON ds.MeterId = me.Id
    WHERE ds.Feeder11Id = :f_id
    AND m.Ts >= DATEADD(hour, -:hrs, @MaxDate)
    ORDER BY m.Ts
""")

params = {"f_id": 1, "hrs": 168}

with engine.connect() as conn:
    df = pd.read_sql(sql_query, conn, params=params)

print(f"Učitano je {len(df)} redova.")
df.head() 

Učitano je 289 redova.


,meter_id,value,timestamp
0,56206,4.067140e+09,2026-04-09 18:30:00.0000000
1,56206,4.067260e+09,2026-04-09 19:00:00.0000000
2,56206,4.067400e+09,2026-04-09 19:30:00.0000000
3,56206,4.067520e+09,2026-04-09 20:00:00.0000000
4,56206,4.067680e+09,2026-04-09 20:30:00.0000000


In [14]:
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [15]:
df.isna().sum()

meter_id     0
value        0
timestamp    0
dtype: int64

In [16]:
print(df.head())

   meter_id         value           timestamp
0     56206  4.067140e+09 2026-04-09 18:30:00
1     56206  4.067260e+09 2026-04-09 19:00:00
2     56206  4.067400e+09 2026-04-09 19:30:00
3     56206  4.067520e+09 2026-04-09 20:00:00
4     56206  4.067680e+09 2026-04-09 20:30:00


In [5]:
sql_query = text("SELECT Id, Name, SsId, MeterId, Feeder33Id, NameplateRating, TsId FROM dbo.Feeders11")

with engine.connect() as conn:
    df_feeders11 = pd.read_sql(sql_query, conn)

print(f"Učitano je {len(df_feeders11)} redova.")
df_feeders11.head(50)

Učitano je 435 redova.


,Id,Name,SsId,MeterId,Feeder33Id,NameplateRating,TsId
0,1,132KV KATAMPE TS_33KV WUSE II FDR_B5_1A,1.0,9.0,2.0,1000.0,NaN
1,2,132KV KATAMPE TS_33KV WUSE II FDR_B5_1B,1.0,6.0,2.0,1000.0,NaN
2,3,132KV KATAMPE TS_33KV WUSE II FDR_B5_2A,1.0,8.0,2.0,1000.0,NaN
3,4,132KV KATAMPE TS_33KV WUSE II FDR_B5_2B,1.0,11.0,2.0,1000.0,NaN
4,5,132KV KATAMPE TS_33KV WUSE II FDR_B5_3A,1.0,7.0,2.0,1000.0,NaN
5,6,132KV KATAMPE TS_33KV WUSE II FDR_B5_3B,1.0,13.0,2.0,1000.0,NaN
6,7,132KV KATAMPE TS_33KV WUSE II FDR_B5_4A,1.0,12.0,2.0,1000.0,NaN
7,8,132KV KATAMPE TS_33KV WUSE II FDR_B5_4B,1.0,10.0,2.0,1000.0,NaN
8,9,132KV KATAMPE TS_33KV WUSE II FDR_B52_1A,2.0,20.0,2.0,1000.0,NaN
9,10,132KV KATAMPE TS_33KV WUSE II FDR_B52_1B,2.0,16.0,2.0,1000.0,NaN


In [6]:
df_feeders11.isna().sum()

Id                   0
Name                 0
SsId                14
MeterId             56
Feeder33Id          15
NameplateRating    260
TsId               421
dtype: int64

In [7]:
invalid_rows = df_feeders11[(df_feeders11['SsId'] == 0) & (df_feeders11['TsId'] == 0)]

double_connected = df_feeders11[df_feeders11['SsId'].notna() & df_feeders11['TsId'].notna()]

both_exist = df_feeders11[
    df_feeders11['SsId'].notna() & (df_feeders11['SsId'] != 0) &
    df_feeders11['TsId'].notna() & (df_feeders11['TsId'] != 0)
]

print(f"SsId=0 i TsId=0: {len(invalid_rows)}")
print(f"SsId != 0 i TsId != 0: {len(double_connected)}")   

SsId=0 i TsId=0: 0
SsId != 0 i TsId != 0: 0


In [8]:
import pandas as pd
import numpy as np

df_feeders11['NameplateRating'] = pd.to_numeric(df_feeders11['NameplateRating'], errors='coerce')
df_feeders11['SsId'] = pd.to_numeric(df_feeders11['SsId'], errors='coerce')
df_feeders11['TsId'] = pd.to_numeric(df_feeders11['TsId'], errors='coerce')

df_feeders11['NameplateRating'] = df_feeders11['NameplateRating'].replace(0, np.nan)

In [ ]:
def make_key(row):
    if pd.notnull(row['SsId']) and row['SsId'] != 0:
        return f"SS_{int(row['SsId'])}"
    elif pd.notnull(row['TsId']) and row['TsId'] != 0:
        return f"TS_{int(row['TsId'])}"
    return None

df_feeders11['TempKey'] = df_feeders11.apply(make_key, axis=1)

In [ ]:
df_feeders11['NameplateRating'] = df_feeders11['NameplateRating'].fillna(
    df_feeders11.groupby('TempKey')['NameplateRating'].transform('mean')
)

preostalo_nan = df_feeders11['NameplateRating'].isnull().sum()
if preostalo_nan > 0:
    print(f"Bilo je {preostalo_nan} stanica bez ijednog podatka. Popunjavam globalnim prosekom.")
    df_feeders11['NameplateRating'] = df_feeders11['NameplateRating'].fillna(df_feeders11['NameplateRating'].mean())

print(f"Konačan broj NaN vrednosti: {df_feeders11['NameplateRating'].isnull().sum()}")

Bilo je 121 stanica bez ijednog podatka. Popunjavam globalnim prosekom.
Konačan broj NaN vrednosti: 0


In [11]:
df_feeders11.isna().sum()

Id                   0
Name                 0
SsId                14
MeterId             56
Feeder33Id          15
NameplateRating      0
TsId               421
PrivremeniKljuc      0
dtype: int64

In [12]:
df_feeders11.shape

(435, 8)